In [1]:
%use kandy
%use dataframe

## Parametros Electricos del Puente
Como primer paso, es necesario fijar el punto de trabajo del convertidor.
Debido a la tension nominal de los capacitores de 470uF (450V), se fija la tension del bus de continua en 400V.
Para la tension de alterna, se decide dejar un margen de seguridad que permita que la tension de red aumente un 20%, sin perder la controlabilidad del equipo.
Para esto se fija la tension de red en 130V, lo que equivale a un indice de modulacion aproximado de m=0.8.

Por otro lado para los parametros electricos del puente de IGBTs en si.
Se considera que durante la conduccion, tanto los IGBTs como los diodos pueden modelarse como fuentes de potencia en serie con una resistencia.<br>
La hoja de datos no entrega estos valores en forma directa, por lo que debemos deducir valores aproximados a partir de los valores de caida de tension cuando circulan 50A de continua y ayundandonos de hojas de datos de IGBTs de corrientes y tensiones maximas similares.
Asi es como se deducen los valores de Vce0 y Vf0, y a partir de ellos rce tanto para el diodo como para el IGBT.

In [2]:
val Vcc = 400                               //V
val Vac_rms = 130                           //V
val Vac_max_rms = 0.577 * Vcc / sqrt(2f)    //V
val m = Vac_rms / Vac_max_rms
val f_sw = 10000                            //Hz
val f_red = 50                              //Hz

val Vce_sat     = 1.65f                     //V
val Vf          = 1.70                      //V
val Vce_bloq    = Vcc                       //V
/*
val Vce0        = 0.85f    //V
val rce_igbt    = (Vce_sat - Vce0) / 50//Ohm
val Vf0         = 0.4428 //V
val rce_diodo   = (1.7 - Vf0) / 50
*/
val Vce0        = 1.08                      //V
val rce_igbt    = (Vce_sat - Vce0) / 50     //Ohm
val Vf0         = 1.3                       //V
val rce_diodo   = (Vf - Vf0) / 50           //Ohm

println("Vce0\t\t= ${String.format("%.2f",Vce0)} V ")
println("rce_igbt\t= ${String.format("%.4f",rce_igbt)} Ohm ")
println("Vf0\t\t\t= ${String.format("%.2f",Vf0)} V ")
println("rce_diodo\t= ${String.format("%.4f",rce_diodo)} Ohm ")
println("m\t\t\t= ${String.format("%.3f", m)}")

Vce0		= 1,08 V 
rce_igbt	= 0,0114 Ohm 
Vf0			= 1,30 V 
rce_diodo	= 0,0080 Ohm 
m			= 0,797


## Formulas para calculo de disipacion de potencia
### Potencia de Encendido/Apagado del IGBT
Se calculan a partir de una situacion de encendido/apagado de referencia, escalada a las condiciones de operacion de nuestro equipo.

In [3]:
fun calcularPotenciaEncendido(il_peak: Double): Double  = f_sw / PI  * 0.0000001448706351 * Vce_bloq * il_peak
fun calcularPotenciaApagado(il_peak: Double): Double    = f_sw / PI  * 0.0000002921576001 * Vce_bloq * il_peak

### Potencia de conduccion
Se utilizan las formulas propuestas en el trabajo "Semiconductor Losses in Voltage Source and Current Source IGBT Converters Based on Analytical Derivation". Si bien trabajamos con modulacion SVPWM, las formulas en cuestion corresponden a modulacion senoidal PWM, que, segun indica el trabajo, sirven como una buena aproximacion.
#### IGBT
Si se considera $\cos(\phi) \approx 1$ resulta:
$$ P_{C,IGBT}=\frac{(\frac{\pi}{4} + \frac{2}{3} m) rce}{2\pi} il_{peak}^2   +   \frac{(1 + \frac{\pi}{4} * m) * Vce0}{2\pi} il_{peak} $$
#### Diodo
Si se considera $\cos(\phi) \approx 1$ resulta:
$$ P_{C,Diodo}=\frac{(\frac{\pi}{4} - \frac{2}{3} m) rce}{2\pi} il_{peak}^2   +   \frac{(1 - \frac{\pi}{4} * m) * Vf0}{2\pi} il_{peak} $$

In [4]:
fun calcularPotenciaConduccionIGBT(il_peak: Double): Double {
    return (PI/4 + 2/3 * m) * rce_igbt * il_peak.pow(2) / (2*PI) + (1 + PI/4 * m) * Vce0 * il_peak / (2*PI)
}

fun calcularPotenciaConduccionDiodo(il_peak: Double): Double {
    return (PI/4 - 2/3 * m) * rce_diodo * il_peak.pow(2) / (2*PI) + (1 - PI/4 * m) * Vf0 * il_peak / (2*PI)
}

## Potencia Total
En el caso del diodo, se considera que solo se disipa potencia durante la conduccion del mismo.
En el caso del IGBT, la potencia total esta compuesta por la potencia de switching y la de conduccion.

In [5]:
fun calcularPotenciaTotalDiodo(il_peak: Double) = calcularPotenciaConduccionDiodo(il_peak)
fun calcularPotenciaTotalIGBT(il_peak: Double) =
    calcularPotenciaConduccionIGBT(il_peak) +
    calcularPotenciaEncendido(il_peak) +
    calcularPotenciaApagado(il_peak)
fun calcularPotenciaTotalPar(ilPeak: Double) = calcularPotenciaTotalDiodo(ilPeak) + calcularPotenciaTotalIGBT(ilPeak)

## Circuito Termico Equivalente
Pudiendo ser capaces ya de calcular la disipacion de potencia en el puente a partir de los parametros del mismo y de la corriente trifasica, podemos calcular la corriente maxima admitida.
La hoja de datos establece 2 limites de temperatura distintos para el integrado:
* Temperatura maxima de capsula: 100 °C
* Temperatura maxima de juntura: 125 °C
### Resistencias Termicas
* #### Juntura-Capsula
  Tambien brinda valores de resistencias termicas capsula-juntura en forma individual para cada Diodo y para cada IGBT.
  $ R_{Qjc} = 0.88 \Omega$
  $ R_{Djc} = 1.78 \Omega$
* #### Capsula-Disipador
   Ademas se sugiere un valor para la resistencia capsula-disipador correspondiente a la pasta termica, el cual se indica por par (1 IGBT + 1 Diodo).
   $ R_{cd} = 0.2 \Omega$ (por cada par)
* #### Disipador-Ambiente
  En cuanto al disipador, es dificil identificar un valor de resistencia termica, debido a que al ser el mismo mucho mas grande que la superficie de contacto del integrado, seria necesario realizar una simulacion termica para estudiar la transmision de calor de la capsula al disipador, a traves de la pasta termica. En consecuencia, se trabajaron con 2 valores de resistencia termica distinta para el disipador, correspondientes al mejor y peor caso. En el mejor caso, se utiliza la resistencia termica correspondiente a la longitud total del disipador (250mm) y en el peor caso, la resistencia correspondiente a la longitud del integrado (80mm).<br>

Se obtienen asi cotas superiores e inferiores para la potencia maxima disipada por el puente.

In [6]:
fun bisectionMethod(
    f: (Double) -> Double,
    a: Double,
    b: Double,
    tolerance: Double = 1e-6,
    maxIterations: Int = 100
): Double {
    require(f(a) * f(b) < 0) { "Function must have opposite signs at interval endpoints" }
    var left = a
    var right = b
    var iteration = 0
    var mid = 0.0
    while (iteration < maxIterations) {
        // Calculate midpoint
        mid = (left + right) / 2
        val fMid = f(mid)
        // Check if we're within tolerance
        if (right - left < 2 * tolerance || fMid == 0.0) return mid
        // Update interval
        if (f(left) * fMid < 0) right = mid else left = mid
        iteration++
    }
    return mid
}

In [16]:
val Tmax =  125
val Tamb = 40

val Rth_jc_igbt     = 0.88
val Rth_jc_diodo    = 1.78
val Rth_cd          = 0.2 / 6
val Rth_da          = 0.45 //Mejor caso: 0.45 - Peor caso: 1
val Rth_ca          = Rth_cd + Rth_da

/* LIMITE DE CORRIENTE SEGUN TEMPERATURA DE JUNTURA MAXIMA */
val ilMax_juntura = bisectionMethod(
    f = { v -> Tamb + (Rth_jc_igbt/6 + Rth_ca) * 6 * calcularPotenciaTotalIGBT(v) + Rth_ca * 6 * calcularPotenciaTotalDiodo(v) - Tmax },
    a = 0.0,
    b = 100.0
)
val Pmax_segun_juntura = 6 * calcularPotenciaTotalIGBT(ilMax_juntura) + 6 * calcularPotenciaTotalDiodo(ilMax_juntura)
println("LIMITACIONES DEBIDO A LA JUNTURA DEL IGBT:")
println("Corriente Maxima (rms): ${String.format("%.2f", ilMax_juntura/sqrt(2f))} A")
println("Potencia Disipada Maxima 1 IGBT: ${String.format("%.2f", calcularPotenciaTotalIGBT(ilMax_juntura))} W")
println("Potencia Disipada Maxima 1 Diodo: ${String.format("%.2f", calcularPotenciaTotalDiodo(ilMax_juntura))} W")
println("Potencia Disipada Maxima 1 Par: ${String.format("%.2f", calcularPotenciaTotalIGBT(ilMax_juntura) + calcularPotenciaTotalDiodo(ilMax_juntura))} W")
println("Potencia Disipada Maxima Total: ${String.format("%.2f", Pmax_segun_juntura)} W")
println("Potencia Nominal del Equipo: ${String.format("%.0f", 3 * Vac_rms * ilMax_juntura / sqrt(2.0))} W")
println("Temperatura de Juntura: ${String.format("%.0f", Tamb + Rth_ca * Pmax_segun_juntura + Rth_jc_igbt * calcularPotenciaTotalIGBT(ilMax_juntura))} °C")
println("Temperatura de Capsula: ${String.format("%.0f", Tamb + Rth_ca * Pmax_segun_juntura)} °C")

/* LIMITE DE CORRIENTE SEGUN TEMPERATURA DE JUNTURA MAXIMA DEL DIODO */
val ilMax_juntura_diodo = bisectionMethod(
    f = { v -> Tamb + (Rth_jc_diodo/6 + Rth_ca) * 6 * calcularPotenciaTotalDiodo(v) + Rth_ca * 6 * calcularPotenciaTotalIGBT(v) - Tmax },
    a = 0.0,
    b = 100.0
)
val Pmax_juntura_diodo = 6 * calcularPotenciaTotalIGBT(ilMax_juntura_diodo) + 6 * calcularPotenciaTotalDiodo(ilMax_juntura_diodo)
println("\nLIMITACIONES DEBIDO A LA JUNTURA DEL DIODO:")
println("Corriente Maxima (rms): ${String.format("%.2f", ilMax_juntura_diodo/sqrt(2f))} A")
println("Potencia Disipada Maxima 1 IGBT: ${String.format("%.2f", calcularPotenciaTotalIGBT(ilMax_juntura_diodo))} W")
println("Potencia Disipada Maxima 1 Diodo: ${String.format("%.2f", calcularPotenciaTotalDiodo(ilMax_juntura_diodo))} W")
println("Potencia Disipada Maxima 1 Par: ${String.format("%.2f", calcularPotenciaTotalIGBT(ilMax_juntura_diodo) + calcularPotenciaTotalDiodo(ilMax_juntura_diodo))} W")
println("Potencia Disipada Maxima Total: ${String.format("%.2f", Pmax_juntura_diodo)} W")
println("Potencia Nominal del Equipo: ${String.format("%.0f", 3 * Vac_rms * ilMax_juntura_diodo / sqrt(2.0))} W")
println("Temperatura de Juntura: ${String.format("%.0f", Tamb + Rth_ca * Pmax_juntura_diodo + Rth_jc_igbt * calcularPotenciaTotalIGBT(ilMax_juntura_diodo))} °C")
println("Temperatura de Capsula: ${String.format("%.0f", Tamb + Rth_ca * Pmax_juntura_diodo)} °C")

/* LIMITE DE CORRIENTE SEGUN TEMPERATURA DE CAPSULA MAXIMA */
val ilMax_capsula = bisectionMethod(
    f = { v -> Tamb + (Rth_ca) * 6 * (calcularPotenciaTotalIGBT(v) + calcularPotenciaTotalDiodo(v)) - 100 },
    a = 0.0,
    b = 100.0
)
val Pmax_capsula = 6 * calcularPotenciaTotalIGBT(ilMax_capsula) + 6 * calcularPotenciaTotalDiodo(ilMax_capsula)
println("\nLIMITACIONES DEBIDO A LA TEMPERATURA DE CAPSULA")
println("Corriente Maxima (rms): ${String.format("%.2f", ilMax_capsula/sqrt(2f))} A")
println("Potencia Disipada Maxima 1 IGBT: ${String.format("%.2f", calcularPotenciaTotalIGBT(ilMax_capsula))} W")
println("Potencia Disipada Maxima 1 Diodo: ${String.format("%.2f", calcularPotenciaTotalDiodo(ilMax_capsula))} W")
println("Potencia Disipada Maxima 1 Par: ${String.format("%.2f", calcularPotenciaTotalIGBT(ilMax_capsula) + calcularPotenciaTotalDiodo(ilMax_capsula))} W")
println("Potencia Disipada Maxima Total: ${String.format("%.2f", 6 * (calcularPotenciaTotalIGBT(ilMax_capsula) + calcularPotenciaTotalDiodo(ilMax_capsula)))} W")
println("Potencia Nominal del Equipo: ${String.format("%.0f", 3 * Vac_rms * ilMax_capsula / sqrt(2.0))} W")
println("Temperatura de Juntura: ${String.format("%.0f", Tamb + Rth_ca * Pmax_capsula + Rth_jc_igbt * calcularPotenciaTotalIGBT(ilMax_capsula))} °C")
println("Temperatura de Capsula: ${String.format("%.0f", Tamb + Rth_ca * Pmax_capsula)} °C")
println("Temperatura de Disipador: ${String.format("%.1f", Tamb + Rth_da * Pmax_capsula)} °C")


LIMITACIONES DEBIDO A LA JUNTURA DEL IGBT:
Corriente Maxima (rms): 16,78 A
Potencia Disipada Maxima 1 IGBT: 20,64 W
Potencia Disipada Maxima 1 Diodo: 2,40 W
Potencia Disipada Maxima 1 Par: 23,05 W
Potencia Disipada Maxima Total: 138,28 W
Potencia Nominal del Equipo: 6546 W
Temperatura de Juntura: 125 °C
Temperatura de Capsula: 107 °C

LIMITACIONES DEBIDO A LA JUNTURA DEL DIODO:
Corriente Maxima (rms): 19,81 A
Potencia Disipada Maxima 1 IGBT: 24,54 W
Potencia Disipada Maxima 1 Diodo: 2,96 W
Potencia Disipada Maxima 1 Par: 27,50 W
Potencia Disipada Maxima Total: 164,98 W
Potencia Nominal del Equipo: 7727 W
Temperatura de Juntura: 141 °C
Temperatura de Capsula: 120 °C

LIMITACIONES DEBIDO A LA TEMPERATURA DE CAPSULA
Corriente Maxima (rms): 15,16 A
Potencia Disipada Maxima 1 IGBT: 18,57 W
Potencia Disipada Maxima 1 Diodo: 2,12 W
Potencia Disipada Maxima 1 Par: 20,69 W
Potencia Disipada Maxima Total: 124,14 W
Potencia Nominal del Equipo: 5911 W
Temperatura de Juntura: 116 °C
Temperatura de 

In [8]:
val corrienteRms = List(51) { index -> 0.0 + index / 50.toDouble() * 20.0}
val tempCapsula = corrienteRms.map { i -> Tamb + Rth_ca * 6 * calcularPotenciaTotalPar(i*sqrt(2.0)) }
val tempJunturaIgbt = corrienteRms.map { i -> Tamb + Rth_ca * 6 * calcularPotenciaTotalPar(i*sqrt(2.0)) + Rth_jc_igbt * calcularPotenciaTotalIGBT(i*sqrt(2.0)) }
val tempJunturaDiodo = corrienteRms.map { i -> Tamb + Rth_ca * 6 * calcularPotenciaTotalPar(i*sqrt(2.0)) + Rth_jc_diodo * calcularPotenciaTotalDiodo(i*sqrt(2.0)) }

val df = dataFrameOf("i" to corrienteRms, "t_caps" to tempCapsula, "t_junt_igbt" to tempJunturaIgbt, "t_junt_diodo" to tempJunturaDiodo)
df

i,t_caps,t_junt_igbt,t_junt_diodo
"0,000000","40,000000","40,000000","40,000000"
"0,400000","43,208069","43,624567","43,286634"
"0,800000","46,425761","47,259559","46,584030"
"1,200000","49,653075","50,904976","49,892187"
"1,600000","52,890011","54,560818","53,211106"
"2,000000","56,136570","58,227085","56,540786"
"2,400000","59,392751","61,903777","59,881229"
"2,800000","62,658555","65,590894","63,232432"
"3,200000","65,933981","69,288435","66,594397"
"3,600000","69,219030","72,996402","69,967124"


In [18]:
run {
    val iLRms = List(21) { i -> i}
    val pCondIgbt   = iLRms.map { i -> calcularPotenciaConduccionIGBT(i.toDouble() * sqrt(2.0))}
    val pCondDiodo  = iLRms.map { i -> calcularPotenciaConduccionDiodo(i.toDouble() * sqrt(2.0))}
    val pOnIgbt     = iLRms.map { i -> calcularPotenciaEncendido(i.toDouble() * sqrt(2.0))}
    val pOffIgbt    = iLRms.map { i -> calcularPotenciaApagado(i.toDouble() * sqrt(2.0))}
    val df = dataFrameOf(
        "il"            to iLRms.map { it.toString() }.toList(),
        "p_cond_igbt"   to pCondIgbt.map { it.toString() }.toList(),
        "p_cond_diodo"  to pCondDiodo.map { it.toString() }.toList(),
        "p_on_igbt"     to pOnIgbt.map { it.toString() }.toList(),
        "p_off_igbt"    to pOffIgbt.map { it.toString() }.toList()
    )
    df.writeCsv("potencias_disipadas.csv")
}

# Plots

In [9]:
val corriente = List(21) { i -> i}
val p_cond_igbt = corriente.map { i -> calcularPotenciaConduccionIGBT(i.toDouble())}
val p_cond_diodo= corriente.map { i -> calcularPotenciaConduccionDiodo(i.toDouble())}
val p_on        = corriente.map { i -> calcularPotenciaEncendido(i.toDouble())}
val p_off       = corriente.map { i -> calcularPotenciaApagado(i.toDouble())}

run {
    val data = mapOf(
        "corriente" to corriente + corriente + corriente + corriente,
        "potencia" to p_cond_igbt + p_cond_diodo + p_on + p_off,
        "legends" to List(corriente.size) { "Conduccion IGBT" } + List(corriente.size) { "Conduccion Diodo" } + List(corriente.size) { "Encendido" } + List(corriente.size) { "Apagado" }
    )  // Combine data into a map
    plot(data) { // Begin plotting
        groupBy("legends") {
            line {
                x("corriente") { axis.name = "Corriente de Pico [A]"}
                y("potencia") { axis.name = "Potencia [W]" }
                color("legends")
            }
            layout { // Set plot layout
                title = "Disipacion de potencia" // Add title
                size = 1300 to 500 // Plot dimension settings
            }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="JqgmCF"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"corriente",
"y":"potencia",
"color":"legends",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"potencia":[0.0,0.28084893984394504,0.5645478795686808,0.8510968191742073,1.1404957586605244,1.4327446980276322,1.727843637275531,2.02579257640422,2.3265915154137002,2.630240454303971,2.9367393930750323,3.2460883317268845,3.5582872702595276,3.873336208672961,4.191235146967185,4.511984085142201,4.835583023198006,5.162031961134602,5.49133089895199,5.823479836650168,6.158478774229136,6.4963277116888944,6.837026649029445,7.180575586250786,7.5269745233529175,7.876223460335839,8.228322397199552,8.583271333944055,8.941070270569348,9.301719207075436,9.665218143462312,0.0,0.07845922067354404,0.15891844134708807,0.24137766202063213,0.32583688269417616,0.41229610336772027,0.5007553240412642,0.5912145447148084,0.6836737653883522,0.7781329860618964,0.8745922067354405,0.9730514274089845,1.0735106480825285,1.1759698687560727,1.2804290894296166,1.3868883101031604,1.4953475307767046,1.6058067514502485,1.7182659721237927,1.8327251927973367,1.949184413470881,2.067643634144425,2.188102854817969,2.3105620754915126,2.435021296165057,2.561480516838601,2.689939737512145,2.820398958185689,2.952858178859233,3.0873173995327767,3.223776620206321,0.0,0.1844550214802179,0.3689100429604358,0.5533650644406537,0.7378200859208716,0.9222751074010894,1.1067301288813074,1.2911851503615253,1.4756401718417431,1.660095193321961,1.8445502148021788,2.029005236282397,2.213460257762615,2.3979152792428327,2.5823703007230505,2.7668253222032684,2.9512803436834862,3.135735365163704,3.320190386643922,3.50464540812414,3.6891004296043577,3.8735554510845756,4.058010472564794,4.242465494045011,4.42692051552523,4.611375537005447,4.795830558485665,4.980285579965883,5.164740601446101,5.3491956229263184,5.533650644406537,0.0,0.37198660974224174,0.7439732194844835,1.1159598292267252,1.487946438968967,1.8599330487112087,2.2319196584534504,2.603906268195692,2.975892877937934,3.3478794876801756,3.7198660974224174,4.091852707164659,4.463839316906901,4.835825926649143,5.207812536391384,5.579799146133626,5.951785755875868,6.32377236561811,6.695758975360351,7.0677455851025925,7.439732194844835,7.811718804587077,8.183705414329317,8.55569202407156,8.927678633813802,9.299665243556044,9.671651853298286,10.043638463040526,10.415625072782769,10.78761168252501,11.159598292267251],
"&merged_groups":["Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion IGBT","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Diodo","Conduccion Dio

In [10]:
val p_total         = p_cond_igbt
    .zip(p_on) { a,b -> a+b }
    .zip(p_off){ a,b -> a+b }
run {
    val data = mapOf(
        "corriente" to corriente,
        "potencia" to p_total,
    )  // Combine data into a map

    plot(data) { // Begin plotting
        hLine {
            yIntercept.constant(calcularPotenciaTotalIGBT(ilMax_capsula))
            color = Color.RED
            type = LineType.DASHED
        }
        line {
            x("corriente") { axis.name = "Corriente de Pico [A]"}
            y("potencia") { axis.name = "Potencia [W]" }
            color = Color.YELLOW
        }
        layout { // Set plot layout
            title = "IGBT - Disipacion de potencia" // Add title
            size = 1300 to 500 // Plot dimension settings
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="EqoRBJ"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"IGBT - Disipacion de potencia"
},
"mapping":{
},
"data":{
"potencia":[0.0,0.8372905710664047,1.6774311420136,2.5204217128415864,3.366262283550363,4.214952854139931,5.066493424610289,5.920883994961438,6.7781245651933775,7.638215135306108,8.501155705299627,9.366946275173941,10.235586844929044,11.107077414564937,11.98141798408162,12.858608553479094,13.73864912275736,14.621539691916416,15.507280260956264,16.3958708298769,17.287311398678327,18.181601967360546,19.078742535923556,19.978733104367357,20.88157367269195,21.78726424089733,22.695804808983503,23.607195376950465,24.52143594479822,25.438526512526764,26.3584670801361],
"corriente":[0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0]
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"yintercept":8.772185156253956,
"color":"#ee6666",
"linetype":"dashed",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"hline",
"data":{
}
},{
"mapping":{
"x":"corriente",
"y":"potencia"
},
"stat":"identity",
"color":"#fac858",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"corriente"
},{
"type":"float",
"column":"potencia"
}]
},
"spec_id":"5"
};
 var containerDiv = document.getElementById("EqoRBJ");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1300.0,
 height: 500.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 <path d="M56.456308594957434 401.8636363636364 L56.456308594957434 401.8636363636364 L94.09384765826239 389.706103235088 L131.73138672156733 377.5071878562619 L169.36892578487232 365.26689022715806 L207.00646484817725 352.98521034777644 L244.64400391148223 340.66214821811707 L282.28154297478716 328.29770383818 L319.9190820380921 315.8918772079651 L357.55662110139707 303.4446683274725 L395.19416016470205 290.95607719670215 L432.831699228007 278.426103815654 L470.4692382913119 265.8547481843281 L508.1067773546169 253.2420103027245 L545.7443164179219 240.5878901708431 L583.3818554812268 227.892387788684 L621.0193945445318 215.15550315624714 L658.6569336078368 202.37723627353247 L696.2944726711418 189.5575871405401 L733.9320117344467 176.69655575726995 L771.5695507977516 163.7941421237221 L809.2070898610566 150.85034623989645 L846.8446289243616 137.86516810579303 L884.4821679876665 124.8386077214119 L922.1197070509714 111.77066508675301 L959.7572461142764 98.6613402018163 L997.3947851775814 85.51063306660194 L1035.0323242408863 72.31854368110976 L1072.6698633041913 59.08507204533987 L1110.307402367496 45.81021815929222 L1147.944941430801 32.493982022966804 L1185.582480494106 19.136363636363626 " fill="none" stroke-width="1.6500000000000001" stroke="rgb(250,200,88)" stroke-opacity="1.0">
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 12 
 
 
 
 

In [11]:
// Function to solve 2x2 linear equation system
fun solveLinearEquation(a1: Double, b1: Double, a2: Double, b2: Double, y1: Double, y2: Double): Pair<Double, Double>? {
    // Calculate determinant
    val det = a1 * b2 - b1 * a2

    // Check if system has a unique solution (det != 0)
    if (Math.abs(det) < 1e-10) {
        println("No unique solution exists (determinant is zero or near zero)")
        return null
    }

    // Cramer's rule
    val x = (y1 * b2 - b1 * y2) / det
    val y = (a1 * y2 - a2 * y1) / det

    return Pair(x, y)
}



In [12]:
//IGBT
val il1 = 20 * sqrt(2.0)
val il2 = 50 * sqrt(2.0)

val a1 = (PI/4 + 2/3 * m) * il1.pow(2) / (2*PI)
val b1 = (1 + PI/4 * m) * il1 / (2*PI)

val a2 = (PI/4 + 2/3 * m) * il2.pow(2) / (2*PI)
val b2 = (1 + PI/4 * m) * il2 / (2*PI)
solveLinearEquation(a1,b1,a2,b2,y1= 8.94, y2= 31.67)
//println("$a1 $b1 3.65")
//println("$a2 $b2 0.23")
50*0.02485 + 0.8734

2.1159

In [13]:
//Diodo
val il1 = 20 * sqrt(2.0)
val il2 = 50 * sqrt(2.0)

val a1 = (PI/4 - 2/3 * m) * il1.pow(2) / (2*PI)
val b1 = (1 - PI/4 * m) * il1 / (2*PI)

val a2 = (PI/4 - 2/3 * m) * il2.pow(2) / (2*PI)
val b2 = (1 - PI/4 * m) * il2 / (2*PI)
solveLinearEquation(a1,b1,a2,b2,y1= 1.88, y2= 6.58)
//0.005013*50+0.8546

(0.005013333333333333, 0.8180587815592754)